## Observação metodológica

A análise em nível municipal é útil para descrever padrões territoriais, mas não permite inferir o voto individual de cada eleitor. Isso é a falácia ecológica. A narrativa final deve manter essa limitação de forma explícita, para acrescentar robustez e credibilidade ao projeto.

## Método para a versão oficial do mapa por candidato

Quando o TSE ou o TRE fornecer a tabela oficial de votos por candidato para presidente em nível municipal, o passo seguinte é:

1. carregar o arquivo em csv/parquet
2. filtrar Goiás
3. normalizar nomes dos municípios
4. fazer o merge com a geografia municipal de Goiás
5. gerar um choropleth por candidato e por ano
6. manter a legenda e a metodologia com clareza

Exemplo de estrutura para esse join:

```python
vote_2022 = pd.read_parquet('data/processed/votos_2022_goias.parquet')
geo = geobr.read_municipality(code_muni='GO', year=2020)
geo['municipio_norm'] = geo['name_muni'].map(normalize_text)
vote_2022['municipio_norm'] = vote_2022['municipio'].map(normalize_text)
final = geo.merge(vote_2022, on='municipio_norm', how='left')
```

Isso permite um mapa por candidato e por ano, com precisão territorial e sem depender de passos improvisados.

In [64]:
from pathlib import Path
import pandas as pd
import geobr
import unicodedata
import plotly.express as px

root = Path('..').resolve()
profile_path = root / 'data' / 'processed' / 'goias_profile_2026.parquet'


def normalize_text(value):
    text = str(value).strip().upper()
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    return text

if profile_path.exists():
    profile = pd.read_parquet(profile_path).copy()
    profile['municipio_norm'] = profile['nm_municipio'].map(normalize_text)
    profile['total_eleitores'] = pd.to_numeric(profile['total_eleitores'], errors='coerce').fillna(0)
    profile['share_feminino'] = (profile['FEMININO'] / profile['total_eleitores'] * 100).fillna(0)
    profile['share_masculino'] = (profile['MASCULINO'] / profile['total_eleitores'] * 100).fillna(0)

    geo = geobr.read_municipality(code_muni='GO', year=2020).copy()
    geo['municipio_norm'] = geo['name_muni'].map(normalize_text)

    bubble = geo.merge(
        profile[['municipio_norm', 'total_eleitores', 'share_feminino', 'share_masculino']],
        on='municipio_norm',
        how='left'
    )
    bubble = bubble.dropna(subset=['total_eleitores']).copy()

    fig = px.scatter(
        bubble,
        x='share_masculino',
        y='share_feminino',
        size='total_eleitores',
        color='total_eleitores',
        hover_name='name_muni',
        hover_data={'name_region': True, 'total_eleitores': ':.0f'},
        size_max=35,
        opacity=0.75,
        title='Goiás: composição por sexo dos municípios',
        labels={
            'share_masculino': '% de eleitores do sexo masculino',
            'share_feminino': '% de eleitores do sexo feminino',
            'total_eleitores': 'Total de eleitores',
            'name_region': 'Região do município'
        },
        color_continuous_scale='Viridis'
    )
    fig.update_layout(
        xaxis_title='% de eleitores do sexo masculino',
        yaxis_title='% de eleitores do sexo feminino',
        legend_title_text='Total de eleitores',
        template='plotly_white',
        margin={'r': 20, 't': 60, 'l': 20, 'b': 20}
    )
    fig.show()

    output = root / 'data' / 'processed' / 'goias_bubble_profile_public.html'
    fig.write_html(output)
    print('Gráfico salvo em:', output)
else:
    print('Base de perfil eleitoral não encontrada.')

Gráfico salvo em: D:\_DataScience_\ELEIÇÕES 2022 e 2026\data\processed\goias_bubble_profile_public.html


In [65]:
from pathlib import Path
import pandas as pd
import geobr
import unicodedata

root = Path('..').resolve()
profile_path = root / 'data' / 'processed' / 'goias_profile_2026.parquet'

# junção com a geografia municipal de Goiás
def normalize_text(value):
    text = str(value).strip().upper()
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    return text

def build_geo_profile_df(profile_df):
    shape = geobr.read_municipality(code_muni='GO', year=2020)
    shape = shape.copy()
    shape['municipio_norm'] = shape['name_muni'].map(normalize_text)
    profile_df = profile_df.copy()
    profile_df['municipio_norm'] = profile_df['nm_municipio'].map(normalize_text)
    merged = shape.merge(
        profile_df[['municipio_norm', 'total_eleitores', 'FEMININO', 'MASCULINO', 'total_secoes']],
        on='municipio_norm',
        how='left'
    )
    merged['share_feminino'] = (merged['FEMININO'] / merged['total_eleitores'] * 100).fillna(0)
    merged['share_masculino'] = (merged['MASCULINO'] / merged['total_eleitores'] * 100).fillna(0)
    return merged

if profile_path.exists():
    geo = build_geo_profile_df(pd.read_parquet(profile_path))
    print(geo.head(3).to_string())
    print('Geometria pronta:', geo.shape)
else:
    print('Base geográfica não pode ser montada sem o perfil municipal de Goiás.')
    print('Base esperada:', profile_path)

   code_muni        name_muni  code_state abbrev_state name_state  code_region   name_region    year                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

## Status da base oficial

A verificação local mostrou que a pasta [Data/2022/TRE-GO](../Data/2022/TRE-GO) contém arquivos de pesquisa eleitoral e não a base oficial de votação por município para presidente em 2022. A estrutura disponível em [Data/2026](../Data/2026) também não mostrou uma tabela oficial pronta para merge municipal.

Conclusão: a etapa oficial de voto por candidato em nível municipal não pode ser completada com os dados que estão disponíveis hoje. O melhor caminho é manter a análise em modo exploratório, com clareza metodológica, até que a base do TSE ou do TRE-GO seja disponibilizada.

In [66]:
def normalize_text(value):
    text = str(value).strip().upper()
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    return text

# procurar indícios da base oficial ao nível municipal
search_roots = [
    root / 'Data' / '2022',
    root / 'Data' / '2026',
    root / 'data' / 'raw',
]

matches = []
for base in search_roots:
    if base.exists():
        for file in base.rglob('*'):
            if file.is_file() and any(token in file.name.lower() for token in ['voto', 'presidente', 'municipio', 'resultado', 'result']) :
                matches.append(file)

print('Arquivos potenciais de resultado eleitoral encontrados:')
for m in matches[:20]:
    print(' -', m.relative_to(root))

if not matches:
    print('Nenhuma base oficial de votos por município/candidato foi identificada em arquivo local.')
    print('O projeto deve seguir em modo exploratório usando perfil do eleitorado e geografia municipal.')

Arquivos potenciais de resultado eleitoral encontrados:
 - Data\2026\Pesquisas Eleitorais - 2026\bairro_municipio_2026.zip


In [67]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import geobr
import plotly.express as px
import unicodedata

root = Path('..').resolve()
print('Workspace root:', root)
print('Dados disponíveis em data:')
for p in sorted((root / 'data').rglob('*'))[:30]:
    print(' -', p.relative_to(root))

profile_path = root / 'data' / 'processed' / 'goias_profile_2026.parquet'
print('\nPerfil disponível?', profile_path.exists())
if profile_path.exists():
    profile = pd.read_parquet(profile_path)
    print('Tamanho do perfil:', profile.shape)
    print(profile.head(2).to_string(index=False))

Workspace root: D:\_DataScience_\ELEIÇÕES 2022 e 2026
Dados disponíveis em data:
 - data\2022
 - data\2022\bu_imgbu_logjez_rdv_vscmr_ele_suplementares_2022.zip
 - data\2022\buweb_ele_suplementares_2022.zip
 - data\2022\ceft_ele_suplementares_2022.zip
 - data\2022\cesp_ele_suplementares_2022.zip
 - data\2022\lista_eleicoes_suplementares_2022.zip
 - data\2022\log_gedai_suplementares_2022.zip
 - data\2022\TRE-GO
 - data\2022\TRE-GO\Pesquisas Eleitorais
 - data\2022\TRE-GO\Pesquisas Eleitorais\evolucao-pesquisa-2026_pais.csv.zip
 - data\2022\TRE-GO\Pesquisas Eleitorais\pesquisa-contratante-2026_pais.csv (1).zip
 - data\2022\TRE-GO\Pesquisas Eleitorais\pesquisa-contratante-2026_pais.csv.zip
 - data\2022\TRE-GO\Pesquisas Eleitorais\pesquisa-contratante_semcargo-2026_pais.csv.zip
 - data\2026
 - data\2026\Candidatos - 2026
 - data\2026\Candidatos - 2026\bem_candidato_2026.zip
 - data\2026\Candidatos - 2026\consulta_cand_2026.zip
 - data\2026\Candidatos - 2026\consulta_cand_complementar_2026.z

# Perfil do eleitorado de Goiás: painel municipal e contexto territorial

Este notebook consolida a análise exploratória do perfil do eleitorado em Goiás com foco em composição demográfica, distribuição municipal e concentração de eleitores. A proposta é apresentar um painel visual legível e publicável, mantendo a transparência metodológica sobre as limitações da base disponível.

Objetivo:
- descrever a estrutura demográfica do eleitorado municipal em Goiás
- gerar um dashboard visual para apresentação pública
- manter a narrativa honesta sobre o que pode e o que não pode ser inferido
- preparar a base e os gráficos para uso em GitHub e LinkedIn

Observação metodológica: a base utilizada neste projeto é o perfil do eleitorado municipal validado para Goiás. Ela não substitui a tabela oficial de votos por candidato do TSE/TRE, e por isso a análise deve ser lida como contexto territorial e demográfico, não como inferência individual de voto.

## Resumo público para GitHub e LinkedIn

Este projeto apresenta um painel exploratório sobre o perfil do eleitorado em Goiás, com foco em como o eleitorado está distribuído entre municípios, por sexo e por faixa etária. A análise é territorial e demográfica: ela descreve a estrutura do eleitorado e a concentração do eleitorado em municípios do estado, sem afirmar o voto individual de cada pessoa.

A base validada neste ambiente foi a estrutura do eleitorado por seção de Goiás, agregada em nível municipal. Essa base permite observar a composição do eleitorado por gênero e juventude e mapear padrões territoriais em escala municipal.

A etapa de votos por candidato para presidente em nível municipal depende da disponibilização oficial do TSE/TRE. Portanto, a leitura correta deste material é de contexto e perfil do eleitorado, não de inferência de voto individual. Em outras palavras, o gráfico é útil para retratar a realidade demográfica do eleitorado, mas não substitui os resultados eleitorais oficiais do processo eleitoral.

In [68]:
from pathlib import Path
import pandas as pd
import geobr
import unicodedata

root = Path('..').resolve()
profile_path = root / 'data' / 'processed' / 'goias_profile_2026.parquet'
profile = pd.read_parquet(profile_path)

# normalização dos nomes para merge geográfico
profile['municipio_norm'] = profile['nm_municipio'].map(lambda x: ''.join(
    ch for ch in unicodedata.normalize('NFKD', str(x).upper()) if not unicodedata.combining(ch)
))

profile['total_eleitores'] = pd.to_numeric(profile['total_eleitores'], errors='coerce').fillna(0)
profile['share_feminino'] = (profile['FEMININO'] / profile['total_eleitores'] * 100).fillna(0)
profile['share_masculino'] = (profile['MASCULINO'] / profile['total_eleitores'] * 100).fillna(0)

shape = geobr.read_municipality(code_muni='GO', year=2020)
shape['municipio_norm'] = shape['name_muni'].map(lambda x: ''.join(
    ch for ch in unicodedata.normalize('NFKD', str(x).upper()) if not unicodedata.combining(ch)
))
geo = shape.merge(profile[['municipio_norm', 'total_eleitores', 'share_feminino', 'share_masculino', 'total_secoes']], on='municipio_norm', how='left')

print('Municípios na base geográfica:', geo.shape[0])
print('Eleitores totais em Goiás:', profile['total_eleitores'].sum())
print('Média de eleitores por município:', round(profile['total_eleitores'].mean(), 2))
print(geo[['name_muni', 'total_eleitores', 'share_feminino']].head(5).to_string(index=False))

Municípios na base geográfica: 246
Eleitores totais em Goiás: 5080755
Média de eleitores por município: 20653.48
         name_muni  total_eleitores  share_feminino
   Abadia de Goiás          15641.0       51.831724
         Abadiânia          12614.0       51.109878
           Acreúna          15990.0       51.019387
         Adelândia           2971.0       48.300236
Água Fria de Goiás           5108.0       46.632733


In [69]:
from pathlib import Path
import pandas as pd
import geobr
import unicodedata

root = Path('..').resolve()
profile_path = root / 'data' / 'processed' / 'goias_profile_2026.parquet'


def normalize_text(value):
    text = str(value).strip().upper()
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    return text

if profile_path.exists():
    profile = pd.read_parquet(profile_path).copy()
    profile['municipio_norm'] = profile['nm_municipio'].map(normalize_text)
    profile['total_eleitores'] = pd.to_numeric(profile['total_eleitores'], errors='coerce').fillna(0)
    profile['share_feminino'] = (profile['FEMININO'] / profile['total_eleitores'] * 100).fillna(0)
    profile['share_masculino'] = (profile['MASCULINO'] / profile['total_eleitores'] * 100).fillna(0)

    geo = geobr.read_municipality(code_muni='GO', year=2020).copy()
    geo['municipio_norm'] = geo['name_muni'].map(normalize_text)

    merged = geo.merge(
        profile[['municipio_norm', 'total_eleitores', 'share_feminino', 'share_masculino']],
        on='municipio_norm',
        how='left'
    ).dropna(subset=['total_eleitores']).copy()

    print('Municípios validados:', len(merged))
    print('Faixa de % feminino:', round(merged['share_feminino'].min(), 2), 'a', round(merged['share_feminino'].max(), 2), '%')
    print('Faixa de % masculino:', round(merged['share_masculino'].min(), 2), 'a', round(merged['share_masculino'].max(), 2), '%')
else:
    print('Base de perfil eleitoral não encontrada.')

Municípios validados: 245
Faixa de % feminino: 45.91 a 55.06 %
Faixa de % masculino: 44.94 a 54.09 %


In [70]:
from pathlib import Path
import numpy as np
import pandas as pd
import geobr
import unicodedata
import plotly.graph_objects as go
from plotly.subplots import make_subplots

root = Path('..').resolve()
profile_path = root / 'data' / 'processed' / 'goias_profile_2026.parquet'


def normalize_text(value):
    text = str(value).strip().upper()
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    return text

if profile_path.exists():
    profile = pd.read_parquet(profile_path).copy()
    profile['municipio_norm'] = profile['nm_municipio'].map(normalize_text)
    profile['total_eleitores'] = pd.to_numeric(profile['total_eleitores'], errors='coerce').fillna(0)
    profile['share_feminino'] = (profile['FEMININO'] / profile['total_eleitores'] * 100).fillna(0)
    profile['share_masculino'] = (profile['MASCULINO'] / profile['total_eleitores'] * 100).fillna(0)

    giovane_cols = [c for c in ['16 a 19 anos', '20 a 24 anos', '25 a 29 anos'] if c in profile.columns]
    if giovane_cols:
        profile['eleitores_16_29'] = profile[giovane_cols].sum(axis=1)
        profile['share_jovens_16_29'] = (profile['eleitores_16_29'] / profile['total_eleitores'] * 100).fillna(0)

    geo = geobr.read_municipality(code_muni='GO', year=2020).copy()
    geo['municipio_norm'] = geo['name_muni'].map(normalize_text)

    dashboard = geo.merge(
        profile[['municipio_norm', 'total_eleitores', 'share_feminino', 'share_masculino', 'share_jovens_16_29']],
        on='municipio_norm',
        how='left'
    ).dropna(subset=['total_eleitores']).copy()

    dashboard['bubble_size'] = np.sqrt(dashboard['total_eleitores']) * 1.75
    top = dashboard.sort_values('total_eleitores', ascending=False).head(15)
    municipal_weight = dashboard.assign(
        eleitorado_percent = dashboard['total_eleitores'] / dashboard['total_eleitores'].sum() * 100
    ).sort_values('eleitorado_percent', ascending=False).head(15)

    fig = make_subplots(
        rows=2,
        cols=3,
        subplot_titles=[
            '1) Sexo por município',
            '2) Distribuição do % feminino',
            '3) Eleitorado x % feminino',
            '4) Perfil feminino x juventude',
            '5) Top 15 municípios',
            '6) Participação municipal no total de eleitores'
        ],
        specs=[[{'type': 'scatter'}, {'type': 'histogram'}, {'type': 'scatter'}], [{'type': 'scatter'}, {'type': 'bar'}, {'type': 'bar'}]],
        horizontal_spacing=0.12,
        vertical_spacing=0.18,
    )

    fig.add_trace(
        go.Scatter(
            x=dashboard['share_masculino'],
            y=dashboard['share_feminino'],
            mode='markers',
            marker=dict(
                size=dashboard['bubble_size'],
                color=dashboard['total_eleitores'],
                colorscale='Viridis',
                showscale=True,
                colorbar=dict(title='Eleitores')
            ),
            text=dashboard['name_muni'],
            hovertemplate='<b>%{text}</b><br>Masculino: %{x:.1f}%<br>Feminino: %{y:.1f}%<br>Eleitores: %{marker.color:,}<extra></extra>',
            showlegend=False,
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Histogram(
            x=dashboard['share_feminino'],
            nbinsx=20,
            marker_color='#4c72b0',
            name='Feminino',
            opacity=0.8,
        ),
        row=1,
        col=2,
    )

    fig.add_trace(
        go.Scatter(
            x=dashboard['total_eleitores'],
            y=dashboard['share_feminino'],
            mode='markers',
            marker=dict(size=8, color='#2ca02c', opacity=0.7),
            text=dashboard['name_muni'],
            hovertemplate='<b>%{text}</b><br>Eleitores: %{x:,}<br>% Feminino: %{y:.1f}%<extra></extra>',
            showlegend=False,
        ),
        row=1,
        col=3,
    )

    fig.add_trace(
        go.Scatter(
            x=dashboard['share_feminino'],
            y=dashboard['share_jovens_16_29'],
            mode='markers',
            marker=dict(
                size=np.sqrt(dashboard['total_eleitores']) * 1.4,
                color=dashboard['total_eleitores'],
                colorscale='Plasma',
                showscale=False,
            ),
            text=dashboard['name_muni'],
            hovertemplate='<b>%{text}</b><br>% Feminino: %{x:.1f}%<br>% Jovens 16-29: %{y:.1f}%<br>Eleitores: %{marker.color:,}<extra></extra>',
            showlegend=False,
        ),
        row=2,
        col=1,
    )

    fig.add_trace(
        go.Bar(
            x=top['name_muni'],
            y=top['total_eleitores'],
            marker_color='#55a868',
            hovertemplate='<b>%{x}</b><br>Eleitores: %{y:,}<extra></extra>',
            showlegend=False,
        ),
        row=2,
        col=2,
    )

    fig.add_trace(
        go.Bar(
            x=municipal_weight['name_muni'],
            y=municipal_weight['eleitorado_percent'],
            marker_color='#8c564b',
            hovertemplate='<b>%{x}</b><br>Participação no total: %{y:.2f}%<extra></extra>',
            showlegend=False,
        ),
        row=2,
        col=3,
    )

    fig.update_xaxes(title_text='% eleitores do sexo masculino', row=1, col=1)
    fig.update_yaxes(title_text='% eleitores do sexo feminino', row=1, col=1)
    fig.update_xaxes(title_text='% feminino', row=1, col=2)
    fig.update_yaxes(title_text='Municípios', row=1, col=2)
    fig.update_xaxes(title_text='Total de eleitores', row=1, col=3)
    fig.update_yaxes(title_text='% feminino', row=1, col=3)
    fig.update_xaxes(title_text='% feminino', row=2, col=1)
    fig.update_yaxes(title_text='% jovens 16-29', row=2, col=1)
    fig.update_xaxes(title_text='Município', row=2, col=2)
    fig.update_yaxes(title_text='Eleitores', row=2, col=2)
    fig.update_xaxes(title_text='Município', row=2, col=3)
    fig.update_yaxes(title_text='Participação no total (%)', row=2, col=3)

    fig.update_layout(
        title='Goiás: painel demográfico do eleitorado municipal',
        template='plotly_white',
        height=1100,
        width=1500,
        margin=dict(l=20, r=20, t=80, b=20),
        legend=dict(title='Legenda')
    )
    fig.show()

    output = root / 'data' / 'processed' / 'goias_dashboard_perfil_public.html'
    fig.write_html(output)
    print('Dashboard salvo em:', output)
    print('Municipios no painel:', len(dashboard))
else:
    print('Base de perfil eleitoral não encontrada para montar o dashboard.')

Dashboard salvo em: D:\_DataScience_\ELEIÇÕES 2022 e 2026\data\processed\goias_dashboard_perfil_public.html
Municipios no painel: 245


## Interpretação e cuidado metodológico

A base desse projeto mostra a distribuição territorial do perfil do eleitorado em Goiás, e não os votos por candidato em nível municipal. Portanto, toda correlação entre perfil socioeconômico e voto deve ser tratada como evidência exploratória de contexto, não como inferência individual.

Esse cuidado é essencial para manter a credibilidade da análise e evitar cair na falácia ecológica. Em outras palavras: uma cidade com maior proporção de eleitores com determinado perfil não implica que cada eleitor desse município tenha votado da mesma forma.

Esse ponto deve aparecer explicitamente no README, na apresentação do notebook e no texto de publicação.

## Conclusão do projeto atual

Com os dados disponíveis hoje, a análise mais sólida e publicável é a que combina:
- perfil do eleitorado em Goiás
- geografia municipal
- comparação territorial por município
- transparência sobre a ausência da base oficial de votos presidenciais no conjunto local

Quando o TSE ou TRE disponibilizar a tabela oficial de votação por município para presidente, o notebook pode ser atualizado para incluir o mapa por candidato e por ano sem mudar a metodologia geral.